# **Data Cleaning**

In [117]:
# Load pandas for data processing and datetime utilities for any date operations.
import pandas as pd

# Set option to display all columns and format float values to two decimal places (to avoid scientific notation).
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [118]:
# Load the raw e-commerce dataset for cleaning.
ecommerce_df = pd.read_csv(r'dataset/raw/ecommerce_dataset_+1m.csv')

## Initial Cleaning

Goals for this section:
- Remove unnecessary / redundant columns
- Inspect data quality (nulls, sample, dtypes)
- Round float columns to 2 decimal places
- Flag logically invalid values

In [119]:
# Display all the columns
ecommerce_df.columns

Index(['order_id', 'order_date', 'order_year', 'order_month', 'order_day',
       'order_hour', 'order_minute', 'order_second', 'is_weekend',
       'order_status', 'return_reason', 'customer_id', 'customer_name',
       'gender', 'age', 'customer_segment', 'country', 'city',
       'customer_loyalty_score', 'total_orders_by_customer',
       'account_creation_date', 'product_id', 'product_name', 'category',
       'sub_category', 'brand', 'product_rating_avg', 'product_reviews_count',
       'stock_quantity', 'unit_price_usd', 'quantity', 'discount_percent',
       'discount_amount_usd', 'total_price_usd', 'cost_usd', 'profit_usd',
       'tax_usd', 'currency', 'payment_method', 'payment_status',
       'installment_plan', 'shipping_method', 'shipping_cost_usd',
       'delivery_days', 'shipping_country', 'warehouse_location',
       'delivery_status', 'rating', 'review_sentiment', 'customer_feedback',
       'coupon_used', 'coupon_code', 'campaign_source', 'device_type',
       'traf

### 1. Remove unnecessary columns

In [120]:
# Eliminate columns that are not relevant to the analysis or contain redundant information.

ecommerce_df = ecommerce_df[
    [
        # TIME & CONTEXT    
        'order_date',
        'order_year',
        'order_month',
        
        # CUSTOMER DEMOGRAPHICS / GEOGRAPHY
        'gender',
        'age',
        'customer_segment',
        'country',
        
        # PRODUCT
        'category',
        'sub_category',
        'unit_price_usd',
        'quantity',
        
        # FINANCIALS
        'discount_percent',
        'total_price_usd',
        'profit_usd',
        'profit_margin_percent',
        
        # PAYMENT & SHIPPING
        'payment_method',
        'shipping_method',
        'shipping_cost_usd',
        'delivery_days',
        'shipping_country',
        
        # CUSTOMER BEHAVIOR
        'rating',
        'customer_loyalty_score',
        'coupon_used',
        'session_duration_minutes',
        'pages_visited',
        'abandoned_cart_before',
        
        # RISK & PERFORMANCE
        'fraud_risk_score',
        'device_type',
        
        # MARKETING
        'campaign_source',
        'traffic_source',
    ]
].copy()

### 2. Rename Columns

In [121]:
ecommerce_df = ecommerce_df.rename(
    columns={
        'total_price_usd' : 'revenue_usd',
        'session_duration_minutes' : 'session_duration_min'
    }
)

### 3. Data quality check

In [122]:
ecommerce_df.info()  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000123 entries, 0 to 1000122
Data columns (total 30 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   order_date              1000123 non-null  object 
 1   order_year              1000123 non-null  int64  
 2   order_month             1000123 non-null  int64  
 3   gender                  1000123 non-null  object 
 4   age                     1000123 non-null  int64  
 5   customer_segment        1000123 non-null  object 
 6   country                 1000123 non-null  object 
 7   category                1000123 non-null  object 
 8   sub_category            1000123 non-null  object 
 9   unit_price_usd          1000123 non-null  float64
 10  quantity                1000123 non-null  int64  
 11  discount_percent        1000123 non-null  int64  
 12  revenue_usd             1000123 non-null  float64
 13  profit_usd              1000123 non-null  float64
 14  pr

In [123]:
# Missing values (%)
print(((ecommerce_df.isnull().sum() / len(ecommerce_df)) * 100).round(2).to_string())

order_date               0.00
order_year               0.00
order_month              0.00
gender                   0.00
age                      0.00
customer_segment         0.00
country                  0.00
category                 0.00
sub_category             0.00
unit_price_usd           0.00
quantity                 0.00
discount_percent         0.00
revenue_usd              0.00
profit_usd               0.00
profit_margin_percent    0.00
payment_method           0.00
shipping_method          0.00
shipping_cost_usd        0.00
delivery_days            0.00
shipping_country         0.00
rating                   0.00
customer_loyalty_score   0.00
coupon_used              0.00
session_duration_min     0.00
pages_visited            0.00
abandoned_cart_before    0.00
fraud_risk_score         0.00
device_type              0.00
campaign_source          0.00
traffic_source           0.00


In [124]:
# Random sample
ecommerce_df.sample(10)

,order_date,order_year,order_month,gender,age,customer_segment,country,category,sub_category,unit_price_usd,quantity,discount_percent,revenue_usd,profit_usd,profit_margin_percent,payment_method,shipping_method,shipping_cost_usd,delivery_days,shipping_country,rating,customer_loyalty_score,coupon_used,session_duration_min,pages_visited,abandoned_cart_before,fraud_risk_score,device_type,campaign_source,traffic_source
187301,2024-12-28 14:39:20.011413,2024,12,Male,70,Regular,United Kingdom,Clothing,Kids Wear,91.36,3,0,274.08,84.54,30.85,PayPal,Next Day,9.93,11,United Kingdom,1,33.70,Yes,59.80,15,Yes,41.90,Desktop,Organic,Social
291321,2025-03-25 16:43:30.741108,2025,3,Male,41,Regular,Germany,Sports,Sports Wear,245.98,2,0,491.96,206.56,41.99,Credit Card,Next Day,0.99,4,Germany,3,16.90,No,37.50,14,No,12.50,Desktop,Email,Social
428144,2025-10-18 08:14:07.318273,2025,10,Female,30,Premium,Belgium,Clothing,Kids Wear,33.98,3,15,86.65,32.14,37.09,Apple Pay,Express,2.59,6,Belgium,1,43.90,No,2.70,17,No,16.30,Tablet,Affiliate,Direct
645420,2025-08-20 00:55:47.467695,2025,8,Female,42,VIP,United States,Home,Furniture,116.91,1,5,111.06,36.48,32.85,Credit Card,Next Day,13.20,7,United States,4,85.40,No,41.80,15,Yes,70.80,Mobile,Facebook,Search
392427,2026-01-26 13:49:20.504076,2026,1,Male,58,Regular,Canada,Clothing,Womens Wear,149.29,5,10,671.81,255.56,38.04,Bank Transfer,Economy,11.80,12,Canada,5,12.90,No,7.10,9,Yes,10.50,Tablet,Email,Referral
918866,2024-12-08 07:48:49.284230,2024,12,Male,68,Premium,Belgium,Health,Fitness,89.65,1,0,89.65,42.64,47.56,Credit Card,Standard,11.63,2,Belgium,1,6.80,No,33.30,4,Yes,32.90,Mobile,Facebook,Social
995520,2024-07-25 07:35:34.381196,2024,7,Female,68,Premium,Germany,Home,Bedding,249.16,5,15,"1,058.93",516.98,48.82,Bank Transfer,Next Day,5.83,7,Germany,1,61.20,No,59.90,2,Yes,11.00,Desktop,Organic,Search
507703,2025-12-11 15:20:24.724999,2025,12,Male,44,Regular,United States,Sports,Outdoor,135.44,4,5,514.67,275.51,53.53,Debit Card,Standard,3.35,7,United States,3,82.20,Yes,7.40,2,No,61.50,Tablet,Google Ads,Email
543244,2025-03-28 15:05:54.279961,2025,3,Female,70,Regular,Australia,Electronics,Laptops,256.65,4,0,"1,026.60",504.80,49.17,Apple Pay,Next Day,24.21,13,Australia,5,91.40,Yes,15.40,1,Yes,35.60,Mobile,Organic,Referral
95882,2024-10-28 14:16:19.318394,2024,10,Male,23,Premium,Netherlands,Sports,Accessories,249.14,2,5,473.37,255.23,53.92,Credit Card,Next Day,13.49,9,Netherlands,5,31.20,Yes,5.50,5,No,79.60,Tablet,Email,Social


### 4. Round float columns to 2 decimal places

In [125]:
# Round float values to two decimal places for cleaner output.
float_cols = ecommerce_df.select_dtypes('float')

for col in float_cols.columns:
    ecommerce_df[col] = ecommerce_df[col].round(2)

### 5. Flag logically invalid values

In [126]:
# 1. Age
invalid_age = ecommerce_df[(ecommerce_df['age'] < 0) | (ecommerce_df['age'] > 120)]

# 2. Quantity
invalid_quantity = ecommerce_df[ecommerce_df['quantity'] <= 0]

# 3. Discount
invalid_discount = ecommerce_df[
    (ecommerce_df['discount_percent'] < 0) | (ecommerce_df['discount_percent'] > 100)
]

# 4. Rating (fix this!)
invalid_rating = ecommerce_df[(ecommerce_df['rating'] < 1) | (ecommerce_df['rating'] > 5)]

# 5. Delivery days
invalid_delivery = ecommerce_df[
    (ecommerce_df['delivery_days'] < 0) | (ecommerce_df['delivery_days'] > 60)
]

# 6. Monetary values
invalid_money = ecommerce_df[
    (ecommerce_df['unit_price_usd'] < 0) |
    (ecommerce_df['revenue_usd'] < 0) |
    (ecommerce_df['shipping_cost_usd'] < 0)
]

# 7. Fraud score
invalid_fraud = ecommerce_df[
    (ecommerce_df['fraud_risk_score'] < 0) | (ecommerce_df['fraud_risk_score'] > 100)
]

# Print summary
print(f'Invalid Age        : {len(invalid_age)}')
print(f'Invalid Quantity   : {len(invalid_quantity)}')
print(f'Invalid Discount   : {len(invalid_discount)}')
print(f'Invalid Rating     : {len(invalid_rating)}')
print(f'Invalid Delivery   : {len(invalid_delivery)}')
print(f'Invalid Money      : {len(invalid_money)}')
print(f'Invalid Fraud      : {len(invalid_fraud)}')

Invalid Age        : 0
Invalid Quantity   : 0
Invalid Discount   : 0
Invalid Rating     : 0
Invalid Delivery   : 0
Invalid Money      : 0
Invalid Fraud      : 0


### 6. Save cleaned dataset

In [127]:
# Save the cleaned dataset to a new CSV file.
ecommerce_df.to_csv("dataset/cleaned/ecommerce_cleaned.csv", index=False)